In [ ]:
with client_month_spend as (

    select
        cc.client_id,
        cc.campaigns_cnt,

        date_trunc('month', cc.first_campaign_dt)::date as first_month_dt,

        date_trunc('month', ch.datetime)::date as month_dt,

        (
            extract(year from age(
                date_trunc('month', ch.datetime),
                date_trunc('month', cc.first_campaign_dt)
            )) * 12

            +

            extract(month from age(
                date_trunc('month', ch.datetime),
                date_trunc('month', cc.first_campaign_dt)
            ))

        )::int as month_shift,

        sum(ch.summ_discounted) as month_spend

    from cvm_sbx.{prefix}_CVMB_24118_client_cohorts cc

    join dm.cheque ch
        on ch.contact_id = cc.client_id
        and ch.operation_type_id = 1
        and ch.summ_discounted > 0

    where cc.first_campaign_dt is not null

    group by
        cc.client_id,
        cc.campaigns_cnt,
        first_month_dt,
        month_dt,
        month_shift

),

filtered_months as (

    select *
    from client_month_spend
    where month_shift between -1 and 4

),

spend_iqr as (

    select
        campaigns_cnt,
        month_shift,

        percentile_cont(0.25)
            within group (order by month_spend) as q1,

        percentile_cont(0.75)
            within group (order by month_spend) as q3

    from filtered_months

    group by
        campaigns_cnt,
        month_shift

),

client_month_spend_clean as (

    select
        f.*

    from filtered_months f

    join spend_iqr iqr
        on f.campaigns_cnt = iqr.campaigns_cnt
        and f.month_shift = iqr.month_shift

    where f.month_spend between
        iqr.q1 - 1.5 * (iqr.q3 - iqr.q1)
        and
        iqr.q3 + 1.5 * (iqr.q3 - iqr.q1)

)

select
    campaigns_cnt,

    month_shift,

    month_dt,

    round(avg(month_spend), 2) as avg_spend_per_client,

    round(
        percentile_cont(0.5)
            within group (order by month_spend),
        2
    ) as median_spend_per_client,

    round(sum(month_spend), 2) as total_spend,

    count(*) as clients_after_cleaning

from client_month_spend_clean

group by
    campaigns_cnt,
    month_shift,
    month_dt

order by
    campaigns_cnt,
    month_shift,
    month_dt;

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


# df = pd.read_csv('rto_cohorts.csv')

df['month_dt'] = pd.to_datetime(df['month_dt'])
df['campaigns_cnt'] = df['campaigns_cnt'].astype(int)

df = df.sort_values([
    'campaigns_cnt',
    'month_shift',
    'month_dt'
])


def format_y_axis():
    plt.gca().yaxis.set_major_formatter(
        mticker.FuncFormatter(
            lambda x, _: f'{x:,.0f}'.replace(',', ' ')
        )
    )


# -----------------------------
# 1. Медианный РТО по когортам
# -----------------------------

plt.figure(figsize=(14, 7))

for cohort in sorted(df['campaigns_cnt'].unique()):

    part = df[df['campaigns_cnt'] == cohort]

    plt.plot(
        part['month_shift'],
        part['median_spend_per_client'],
        marker='o',
        linewidth=2,
        label=f'{cohort} камп.'
    )

plt.axvline(
    x=0,
    linestyle='--',
    alpha=0.5
)

plt.title('Медианный РТО по когортам')
plt.xlabel('Месяц относительно первой кампании')
plt.ylabel('Медианный РТО')
plt.grid(True, alpha=0.3)

plt.xticks(
    [-1, 0, 1, 2, 3, 4],
    [
        '-1 мес',
        '1-я кампания',
        '+1 мес',
        '+2 мес',
        '+3 мес',
        '+4 мес'
    ]
)

plt.legend(title='Кол-во кампаний')

format_y_axis()

plt.tight_layout()
plt.show()


# -----------------------------
# 2. Средний РТО по когортам
# -----------------------------

plt.figure(figsize=(14, 7))

for cohort in sorted(df['campaigns_cnt'].unique()):

    part = df[df['campaigns_cnt'] == cohort]

    plt.plot(
        part['month_shift'],
        part['avg_spend_per_client'],
        marker='o',
        linewidth=2,
        label=f'{cohort} камп.'
    )

plt.axvline(
    x=0,
    linestyle='--',
    alpha=0.5
)

plt.title('Средний РТО по когортам')
plt.xlabel('Месяц относительно первой кампании')
plt.ylabel('Средний РТО')
plt.grid(True, alpha=0.3)

plt.xticks(
    [-1, 0, 1, 2, 3, 4],
    [
        '-1 мес',
        '1-я кампания',
        '+1 мес',
        '+2 мес',
        '+3 мес',
        '+4 мес'
    ]
)

plt.legend(title='Кол-во кампаний')

format_y_axis()

plt.tight_layout()
plt.show()


# --------------------------------------
# 3. Среднее vs медиана внутри когорты
# --------------------------------------

for cohort in sorted(df['campaigns_cnt'].unique()):

    part = df[df['campaigns_cnt'] == cohort]

    plt.figure(figsize=(12, 6))

    plt.plot(
        part['month_shift'],
        part['avg_spend_per_client'],
        marker='o',
        linewidth=2,
        label='Среднее'
    )

    plt.plot(
        part['month_shift'],
        part['median_spend_per_client'],
        marker='o',
        linewidth=2,
        label='Медиана'
    )

    plt.axvline(
        x=0,
        linestyle='--',
        alpha=0.5
    )

    plt.title(
        f'РТО - когорта {cohort} камп.'
    )

    plt.xlabel('Месяц относительно первой кампании')
    plt.ylabel('РТО')

    plt.grid(True, alpha=0.3)

    plt.xticks(
        [-1, 0, 1, 2, 3, 4],
        [
            '-1 мес',
            '1-я кампания',
            '+1 мес',
            '+2 мес',
            '+3 мес',
            '+4 мес'
        ]
    )

    plt.legend()

    format_y_axis()

    plt.tight_layout()
    plt.show()